# Занятие 1. От LLM к одному шагу агента
Вы соберёте проверяемый шаг: **задача → предложение действия → проверка → инструмент → наблюдение**. Вы научитесь отделять ответ модели от факта исполнения и результата внешней проверки.

**Что нужно знать:** функции Python, списки, словари, `if`, исключения. JSON разберём на примере; знание нейросетевых библиотек не требуется.

| Работа | Баллы |
|---|---:|
| 1. `make_messages` | 1 |
| 2. `parse_action` | 3 |
| 3. `run_lookup` | 2 |
| 4. `verify_answer` | 1 |
| Разбор трёх трасс ошибок | 1 |
| Заполненный отчёт и объяснение границ автономности | 2 |
| **Всего** | **10** |

Оценка кода зависит от контрактов и тестов, а не от качества LLM. Live-эксперимент можно отметить как невыполненный при недоступном API/GPU; в mock нужен разбор фиксированных трасс и честный отчёт.

## 0. Сначала идея, затем контракт
**Интуиция.** Модель может написать «аудитория 204», не проверяя расписание. Надёжнее позволить ей выбрать поиск, исполнить его обычным кодом и проверить полученный результат.

**Определение.** Обозначим задачу и доступное описание среды через $x$, предложение модели — через $a = \pi(x)$, проверку допустимости — через $V(a)$, исполнение инструмента — через $o = T(a, S)$, где $S$ — данные среды. Инструмент исполняется только при $V(a)=\mathrm{True}$. Оценщик $J(o,y)$ сравнивает наблюдение с эталоном $y$.

Сегодня реализуем **один шаг**, без планирования и повторных действий. В полном цикле наблюдение возвращается модели, а код контролирует продолжение и остановку. Эталон находится у оценщика и не передаётся модели.

**Пример.** Задача — узнать аудиторию. Доступно описание `room — аудитория`, но само значение скрыто в словаре. Предложение `lookup("room")` извлекает значение. Напечатанное моделью предложение ещё ничего не исполнило.

In [52]:
# Готовая демонстрация, без LLM и без API. Предложение написано заранее.
demo_store = {"room": "204", "time": "18:00"}
demo_action = {"tool": "lookup", "args": {"key": "room"}}
print("1. Предложение (scripted):", demo_action)
# Для этой конкретной демонстрации схема уже известна и проверена вручную.
demo_value = demo_store[demo_action["args"]["key"]]
print("2. Инструмент вернул:", demo_value)
print("3. Внешняя проверка:", demo_value == "204")

1. Предложение (scripted): {'tool': 'lookup', 'args': {'key': 'room'}}
2. Инструмент вернул: 204
3. Внешняя проверка: True


**Проверьте себя до продолжения.** Какая строка обращается к среде? Что произойдёт при ключе `building`, которого нет? Почему эта демонстрация не доказывает качество модели?

Далее вы замените частный пример функциями с явными контрактами: безопасно разберёте JSON и обработаете отсутствующий ключ.

## 1. Подключение модели
Сначала оставьте `BACKEND = "mock"`: этот режим позволяет выполнить все обязательные задачи без ключа и загрузки весов. Live-эксперимент в конце выполняется при доступной инфраструктуре; отсутствие API/GPU не снижает оценку за код.

Режимы: `mock` — заранее подготовленные ответы для проверки Python; `local` — веса в вычислительной среде Colab; `openrouter` — запрос внешнему провайдеру. Вне Colab `local` использует ту машину, где работает kernel. Не отправляйте секреты и личные данные.

Для local в Colab включите GPU: «Среда выполнения → Сменить среду выполнения». GPU может быть недоступен. Qwen3-0.6B — небольшая учебная модель, её качество не гарантирует решение задачи.

Для OpenRouter создайте ключ и сохраните его в Colab Secrets под именем `OPENROUTER_API_KEY`, разрешив этому блокноту доступ к секрету. Не вставляйте ключ в код. Сначала выполните конфигурацию и каталог, затем скопируйте выбранный `:free` id в `MODEL_ID`, повторите конфигурацию и создайте клиент. Не нужно повторно выполнять установку зависимостей.

Если модель недоступна, выполняйте задачи в `mock`. Подключение модели можно повторить позже.

In [53]:
BACKEND = "local"  # mock | local | openrouter
MODEL_ID = None  # Для OpenRouter скопируйте id из каталога ниже
MAX_CALLS = 12

In [54]:
if BACKEND == "local":
    %pip -q install "transformers==4.56.2" "accelerate==1.10.1"
# torch уже установлен в обычной среде Colab; локально установите его отдельно.

### Готовый модуль
Эта свёрнутая ячейка — инфраструктура, её не нужно переписывать. Она ограничивает число вызовов, читает ключ без сохранения в notebook и допускает только модели, прошедшие проверку бесплатного каталога. Ограничение вызовов действует на текущий экземпляр клиента; это учебный бюджет, а не защита от намеренного обхода.

In [55]:
"""Учебный доступ к LLM: стандартная библиотека для OpenRouter, lazy local backend."""
import json
import os
import time
from decimal import Decimal, InvalidOperation
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_URL = 'https://openrouter.ai/api/v1'

class ModelAccessError(RuntimeError):
    pass

def _request(path, key=None, payload=None):
    headers = {'Content-Type': 'application/json'}
    if key:
        headers['Authorization'] = 'Bearer ' + key
    req = Request(BASE_URL + path, headers=headers,
                  data=None if payload is None else json.dumps(payload).encode())
    try:
        with urlopen(req, timeout=45) as response:
            result = json.load(response)
    except (json.JSONDecodeError, UnicodeDecodeError):
        raise ModelAccessError('API вернул некорректный JSON. Повторите позже; ответ не выводится.') from None
    except HTTPError as exc:
        hints = {401: 'Проверьте API-ключ.', 402: 'Запрос отклонён по балансу; платного fallback нет.',
                 403: 'Проверьте доступ и настройки аккаунта.',
                 429: 'Лимит запросов; подождите или используйте local/mock.'}
        raise ModelAccessError(f'HTTP {exc.code}. ' + hints.get(exc.code, 'Ошибка провайдера; повторите позже.')) from None
    except (URLError, TimeoutError, OSError):
        raise ModelAccessError('Сеть недоступна или истёк таймаут.') from None
    if not isinstance(result, dict):
        raise ModelAccessError('API вернул JSON неожиданного формата: ожидается объект.')
    if 'error' in result:
        raise ModelAccessError('API вернул ошибку. Ответ и ключ не выводятся.')
    return result

def is_free_text_model(model):
    """Консервативный фильтр: :free и нулевые опубликованные цены."""
    if not isinstance(model, dict):
        return False
    pricing = model.get('pricing') or {}
    model_id = model.get('id')
    if not isinstance(model_id, str) or not model_id.endswith(':free'):
        return False
    if not isinstance(pricing, dict) or not all(k in pricing for k in ('prompt', 'completion')):
        return False
    try:
        if any(Decimal(str(value)) != 0 for value in pricing.values()):
            return False
    except (InvalidOperation, ValueError, TypeError):
        return False
    arch = model.get('architecture') or {}
    if not isinstance(arch, dict):
        return False
    if not all(isinstance(arch.get(k), list) for k in ('input_modalities', 'output_modalities')):
        return False
    return 'text' in arch.get('input_modalities', []) and 'text' in arch.get('output_modalities', [])

def free_models(require_tools=False):
    data = _request('/models').get('data')
    if not isinstance(data, list):
        raise ModelAccessError('Каталог моделей имеет неожиданный формат. Повторите позже.')
    return sorted([m for m in data if is_free_text_model(m)
                   and (not require_tools or 'tools' in (m.get('supported_parameters') or []))],
                  key=lambda m: m['id'])

def read_key():
    key = os.environ.get('OPENROUTER_API_KEY')
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get('OPENROUTER_API_KEY')
        except Exception:
            pass
    if not key:
        from getpass import getpass
        key = getpass('OpenRouter API key (скрытый ввод): ').strip()
    if not key:
        raise ModelAccessError('API-ключ не задан.')
    return key

class ModelClient:
    def __init__(self, backend='mock', model_id=None, key=None, max_calls=12):
        if backend not in ('mock', 'openrouter', 'local'):
            raise ValueError('backend: mock, openrouter или local')
        self.backend, self.model_id, self.key = backend, model_id, key
        self.max_calls, self.calls = max_calls, 0
        self.records = []
        self.tokenizer = self.model = None
        if backend == 'openrouter':
            if model_id not in {m['id'] for m in free_models()}:
                raise ModelAccessError('Выберите модель из текущего бесплатного каталога.')
            self.key = key or read_key()
        if backend == 'local':
            self.model_id = model_id or 'Qwen/Qwen3-0.6B'

    def _load_local(self):
        try:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
        except ImportError:
            raise ModelAccessError('Для local нужны torch и transformers. Выполните ячейку установки.') from None
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
            model.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
        except OSError:
            raise ModelAccessError('Не удалось загрузить локальную модель. Проверьте сеть и model_id.') from None
        self.tokenizer, self.model = tokenizer, model

    def chat(self, messages, max_tokens=128):
        if self.calls >= self.max_calls:
            raise ModelAccessError('Исчерпан учебный бюджет вызовов. Не перезапускайте ячейку ради обхода лимитов провайдера.')
        if not 1 <= max_tokens <= 512:
            raise ValueError('Учебный лимит: 1–512 выходных токенов.')
        self.calls += 1
        started = time.monotonic()
        status = 'error'
        try:
            if self.backend == 'mock':
                answer = '[MOCK: это заглушка, не ответ LLM]'
            elif self.backend == 'openrouter':
                # Повторная проверка каталога перед каждым вызовом; без auto-router и fallback.
                if self.model_id not in {m['id'] for m in free_models()}:
                    raise ModelAccessError('Модель больше не прошла проверку бесплатности.')
                result = _request('/chat/completions', self.key, {
                    'model': self.model_id, 'messages': messages, 'max_tokens': max_tokens,
                    'provider': {'allow_fallbacks': False}})
                choices = result.get('choices')
                if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
                    raise ModelAccessError('API вернул ответ без списка choices. Повторите позже.')
                message = choices[0].get('message')
                if not isinstance(message, dict):
                    raise ModelAccessError('API вернул ответ без объекта message. Повторите позже.')
                answer = message.get('content')
            else:
                if self.model is None:
                    self._load_local()
                import torch
                text = self.tokenizer.apply_chat_template(messages, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
                inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)
                if inputs['input_ids'].shape[1] > 2048:
                    raise ModelAccessError('Для учебного local-режима сократите вход до 2048 токенов.')
                with torch.inference_mode():
                    output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
                answer = self.tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            if not isinstance(answer, str) or not answer.strip():
                raise ModelAccessError('Нет текстового ответа; возможно, исчерпан лимит генерации.')
            status = 'ok'
            return answer
        finally:
            self.records.append({'backend': self.backend, 'model': self.model_id,
                                 'seconds': round(time.monotonic() - started, 3), 'status': status})

    def probe(self):
        try:
            answer = self.chat([{'role': 'user', 'content': 'Reply with the single word OK.'}], max_tokens=64)
            return {'status': 'MOCK_ONLY' if self.backend == 'mock' else 'TEXT_RESPONSE_RECEIVED',
                    'exact_ok': answer.strip() == 'OK', 'answer': answer,
                    'note': 'Не проверяет качество агента или native tool calling.'}
        except ModelAccessError as exc:
            return {'status': 'FAILED', 'reason': str(exc)}


In [56]:
import platform
print("Python:", platform.python_version())
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет; local будет медленнее")
except ImportError:
    print("torch не установлен; OpenRouter и mock доступны без него")
if BACKEND == "openrouter":
    catalog = free_models()
    for m in catalog:
        print(m["id"], "context:", m.get("context_length"), "tools (заявлено):", "tools" in (m.get("supported_parameters") or []))
    print("Наличие в каталоге не гарантирует успешный запрос. Выберите MODEL_ID выше.")

Python: 3.13.15
GPU: Tesla T4


In [57]:
if BACKEND == "openrouter" and not MODEL_ID:
    raise ValueError("Скопируйте id из каталога в MODEL_ID и повторите ячейку конфигурации.")
client = None
try:
    client = ModelClient(BACKEND, MODEL_ID, max_calls=MAX_CALLS)
    print(client.probe())
except ModelAccessError as exc:
    print({"status": "FAILED", "reason": str(exc)})
# При неудачной диагностике продолжайте обязательные задачи без модели.
# Для интеграции выберите mock и повторите конфигурацию и эту ячейку.

{'status': 'TEXT_RESPONSE_RECEIVED', 'exact_ok': True, 'answer': 'OK', 'note': 'Не проверяет качество агента или native tool calling.'}


`TEXT_RESPONSE_RECEIVED` означает только успешный текстовый запрос сейчас. `exact_ok=False` — нарушение запрошенного формата. `MOCK_ONLY` означает запуск заглушки. Поле `tools` в каталоге — заявление API, а не проверка вызова инструментов.

Диагностический запрос тоже расходует квоту. При 429 подождите или используйте local/mock; не перебирайте модели ради обхода лимита. Даже после успешной диагностики следующий запрос может завершиться ошибкой.

## 2. Задача 1: вход модели · 1 балл
**Интуиция.** Программа должна явно передать модели задачу и данные; содержимое переменных Python само по себе модели не видно.

**Контракт.** `make_messages(task: str, context: str)` возвращает ровно два словаря: system и user. Оба `content` — строки; system содержит непустую инструкцию отвечать по данным и признавать их отсутствие. В user должны полностью присутствовать исходные `task` и `context`, включая пробелы и переводы строк. Не изменяйте эти строки.

**Пример.** Для задачи «Когда занятие?» и контекста «Начало в 18:00» оба фрагмента должны попасть в user-сообщение. Способ оформления выберите сами.

**Перед кодом:** почему контекст из внешнего документа не следует автоматически делать system-инструкцией?

In [58]:
def make_messages(task, context):
    system_instruction = "Отвечай строго по предоставленным данным. Если ответа в данных нет, признай их отсутствие."

    user_message = f"Контекст:\n{context}\n\nЗадача:\n{task}"

    return [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_message}
    ]

In [59]:
task_input = "  Когда занятие?\nУкажи время.  "
context_input = "Начало в 18:00\nАудитория: 204"
m = make_messages(task_input, context_input)
assert isinstance(m, list) and len(m) == 2
assert [x["role"] for x in m] == ["system", "user"]
assert all(isinstance(x["content"], str) for x in m)
assert m[0]["content"].strip()
assert task_input in m[1]["content"] and context_input in m[1]["content"]
assert make_messages("", "")[0]["content"].strip()
print("Задача 1: открытые проверки пройдены")

Задача 1: открытые проверки пройдены


## 3. Задача 2: допустимое действие · 3 балла
**Интуиция.** Корректный JSON может содержать неподходящую команду. Сначала разбираем текст, затем проверяем структуру и разрешённые значения.

**Контракт.** `parse_action(text: str)` принимает только объект `{"tool":"lookup","args":{"key":"..."}}`. У объекта ровно поля `tool`, `args`; у `args` ровно поле `key`; `key` — строка. При нарушении выбрасывайте `ValueError`. `json.JSONDecodeError` уже является подклассом `ValueError`. Не используйте `eval`.

**Примеры.** `{"tool":"lookup","args":{"key":"room"}}` допустим. `{"tool":"lookup","args":{"key":3}}` — корректный JSON, но недопустимое действие. Отсутствующий в словаре строковый ключ допустим: это проверяет исполнитель на следующем шаге.

**Перед кодом:** чем неправильный формат действия отличается от неудачного результата инструмента?

In [60]:
def parse_action(text):
    data = json.loads(text)

    if not isinstance(data, dict):
        raise ValueError("Root object must be a dictionary.")

    if set(data.keys()) != {"tool", "args"}:
        raise ValueError("Object must contain exactly 'tool' and 'args' keys.")

    if data["tool"] != "lookup":
        raise ValueError("Tool value must be 'lookup'.")

    if not isinstance(data["args"], dict):
        raise ValueError("'args' must be a dictionary.")

    if set(data["args"].keys()) != {"key"}:
        raise ValueError("'args' object must contain exactly the 'key' field.")

    if not isinstance(data["args"]["key"], str):
        raise ValueError("'key' value must be a string.")

    return data


In [61]:
assert parse_action('{"tool":"lookup","args":{"key":"room"}}') == {"tool":"lookup","args":{"key":"room"}}
assert parse_action('{"tool":"lookup","args":{"key":""}}')["args"]["key"] == ""
invalid_actions = [
    "не JSON", "[]", "null", "{}", '{"tool":"lookup"}',
    '{"tool":"shell","args":{"key":"x"}}',
    '{"tool":"lookup","args":null}', '{"tool":"lookup","args":{}}',
    '{"tool":"lookup","args":{"key":3}}', '{"tool":"lookup","args":{"key":null}}',
    '{"tool":"lookup","args":{"key":"x","extra":1}}',
    '{"tool":"lookup","args":{"key":"x"},"extra":1}'
]
for bad in invalid_actions:
    try:
        parse_action(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Недопустимое действие принято: " + bad)
print("Задача 2: открытые проверки пройдены")

Задача 2: открытые проверки пройдены


## 4. Задача 3: исполнение инструмента · 2 балла
**Интуиция.** «Ключа нет» и «ключ есть, но значение неизвестно» — разные наблюдения.

**Контракт.** `run_lookup(action, store)` получает уже проверенное действие и словарь. Возвращает `{"found": bool, "value": ...}`. Для отсутствующего ключа: `found=False, value=None`. Для существующего ключа со значением `None`: `found=True, value=None`. Не изменяйте `action` и `store`.

**Пример.** В `{"room":"204", "teacher":None}` поиск `teacher` успешен как поиск ключа; поиск `building` сообщает отсутствие ключа.

**Перед кодом:** достаточно ли проверять `store.get(key) is not None`?

In [62]:
def run_lookup(action, store):
    key = action["args"]["key"]

    found = key in store
    value = store[key] if found else None

    return {
        "found": found,
        "value": value
    }

In [63]:
import copy
store = {"room": "204", "unknown": None, "zero": 0, "empty": ""}
a = lambda k: {"tool":"lookup", "args":{"key":k}}
store_before = copy.deepcopy(store)
action = a("room")
action_before = copy.deepcopy(action)
assert run_lookup(action, store) == {"found":True,"value":"204"}
assert run_lookup(a("absent"), store) == {"found":False,"value":None}
assert run_lookup(a("unknown"), store) == {"found":True,"value":None}
assert run_lookup(a("zero"), store) == {"found":True,"value":0}
assert run_lookup(a("empty"), store) == {"found":True,"value":""}
assert store == store_before and action == action_before
print("Задача 3: открытые проверки пройдены")

Задача 3: открытые проверки пройдены


## 5. Задача 4: внешняя проверка · 1 балл
**Интуиция.** «Я уверен» — часть сгенерированного текста, а не свидетельство решения.

**Контракт.** `verify_answer(answer, expected)` возвращает результат равенства $N(answer)=N(expected)$, где $N(x)=\operatorname{strip}(\operatorname{str}(x))$. Это точная учебная метрика для коротких значений, не универсальная оценка произвольного текста.

**Примеры.** `" 204 "` и `204` совпадают после нормализации; `"Уверен, что 204"` и `"204"` не совпадают.

**Перед кодом:** почему нельзя проверять только наличие строки `"204"` внутри ответа?

In [64]:
def verify_answer(answer, expected):
    return str(answer).strip() == str(expected).strip()

In [65]:
assert verify_answer(" 204 ", "204")
assert verify_answer(204, "204")
assert verify_answer("\n204\t", 204)
assert not verify_answer("205", "204")
assert not verify_answer("Уверен, что 204", "204")
assert not verify_answer("2040", 204)
print("Задача 4: открытые проверки пройдены")

Задача 4: открытые проверки пройдены


## 6. Предскажите результат трёх трасс · 1 балл
До запуска заполните в следующей текстовой ячейке прогноз: `json_valid`, `action_valid`, `found`, `result_correct`.

1. `не JSON`.
2. `{"tool":"lookup","args":{"key":"building"}}` — такого ключа нет.
3. `{"tool":"lookup","args":{"key":"room"}}` — ключ есть, значение совпадает с эталоном.

**Проверка понимания:** где должна остановиться каждая трасса? Почему отсутствие ключа не является ошибкой JSON? Ниже готовый стенд, который использует ваши четыре функции. Здесь нет вызовов LLM.

**Мой прогноз и объяснение:**

---
1. не JSON: False, False, False, False

2. "building": True, True, False, False

3. "room": True, True, True, True.

JSON - это синтаксис, он не проверяет содержиоме на правильность логики данных.

In [66]:
def evaluate_proposal(raw, store, expected_key, expected_answer):
    """Готовый стенд: отдельно проверяет формат, действие и результат."""
    result = {"json_valid": False, "action_valid": False, "selected_key": None,
              "key_correct": None, "found": None, "result_correct": False, "error": None}
    try:
        json.loads(raw)
        result["json_valid"] = True
        action = parse_action(raw)
        result["action_valid"] = True
        result["selected_key"] = action["args"]["key"]
        result["key_correct"] = action["args"]["key"] == expected_key
        observation = run_lookup(action, store)
        result["found"] = observation["found"]
        result["result_correct"] = observation["found"] and verify_answer(observation["value"], expected_answer)
    except ValueError as exc:
        result["error"] = type(exc).__name__
    return result

In [67]:
trace_inputs = ["не JSON", '{"tool":"lookup","args":{"key":"building"}}',
                '{"tool":"lookup","args":{"key":"room"}}']
for raw in trace_inputs:
    print(raw, "→", evaluate_proposal(raw, {"room":"204"}, "room", "204"))

не JSON → {'json_valid': False, 'action_valid': False, 'selected_key': None, 'key_correct': None, 'found': None, 'result_correct': False, 'error': 'JSONDecodeError'}
{"tool":"lookup","args":{"key":"building"}} → {'json_valid': True, 'action_valid': True, 'selected_key': 'building', 'key_correct': False, 'found': False, 'result_correct': False, 'error': None}
{"tool":"lookup","args":{"key":"room"}} → {'json_valid': True, 'action_valid': True, 'selected_key': 'room', 'key_correct': True, 'found': True, 'result_correct': True, 'error': None}


## 7. Один шаг с моделью и журнал попыток
Теперь модель знает **описания ключей**, но не значения словаря. Эталон передаётся только проверке.

Формат действия указан через `KEY`, поэтому готовый правильный ответ не содержится в запросе. В mock используется явно подписанная фиксированная строка; это демонстрация механики, не модельный эксперимент.

Сначала выполните одну попытку. Затем при желании измените формулировку запроса и выполните ещё до двух попыток. **Всего не более трёх**, включая неудачные. Технический успех запроса, корректность JSON и правильность результата записываются отдельно. Не сравнивайте модели по этим трём примерам.

Готовая оболочка `MiniAgent` объединяет ваши функции: `messages` формирует вход, `step` проводит одно предложенное действие через проверки и инструмент. Все переходы пока заданы кодом; новые методы реализовывать не нужно. Сначала проверим этот простой шаг, а циклы и остановку добавим на следующих занятиях.

In [68]:
class MiniAgent:
    """Готовая оболочка одного шага; схема и оценщик остаются явными."""
    def __init__(self, store):
        self.store = store

    def messages(self, task, context):
        return make_messages(task, context)

    def step(self, raw, expected_key, expected_answer):
        return evaluate_proposal(raw, self.store, expected_key, expected_answer)


In [69]:
experiment_records = []  # Не сбрасывайте журнал между попытками.
experiment_store = {"room": "204", "time": "18:00", "teacher": "А. Иванова"}
agent = MiniAgent(experiment_store)
tool_context = (
    'Доступен только lookup(key). Ключи: room — аудитория; time — начало занятия; '
    'teacher — преподаватель. Значения можно узнать только через инструмент. '
    'Формат ответа: {"tool":"lookup","args":{"key":"KEY"}}, где KEY — выбранный ключ. '
    'Верни только JSON, без Markdown.'
)

def record_attempt(task, context, *, scripted_response=None):
    if len(experiment_records) >= 3:
        raise ValueError("Три попытки уже записаны. Завершите отчёт.")
    row = {"attempt": len(experiment_records) + 1, "backend": BACKEND,
           "model_id": None if client is None else client.model_id,
           "response_source": "scripted" if BACKEND == "mock" else "model",
           "task": task, "context": context, "raw_response": None,
           "response_received": False, "json_valid": None, "action_valid": None,
           "selected_key": None, "key_correct": None, "found": None,
           "result_correct": False, "error": None}
    try:
        if BACKEND == "mock":
            if not isinstance(scripted_response, str):
                raise ModelAccessError("Для mock передайте заранее записанный scripted_response.")
            raw = scripted_response
        else:
            if client is None:
                raise ModelAccessError("Клиент не создан: завершите диагностику или выберите mock.")
            if client.backend != BACKEND:
                raise ModelAccessError("BACKEND и клиент различаются. Повторите создание клиента.")
            # Вызов находится внутри try: 429/таймаут попадёт в журнал.
            raw = client.chat(agent.messages(task, context))
        row["raw_response"] = raw
        row["response_received"] = True
        row.update(agent.step(raw, "room", "204"))
    except ModelAccessError as exc:
        row["error"] = str(exc)
    experiment_records.append(row)
    return row

In [70]:
record_attempt(
    "Найди номер аудитории следующего занятия.", tool_context,
    scripted_response='{"tool":"lookup","args":{"key":"room"}}'
)

{'attempt': 1,
 'backend': 'local',
 'model_id': 'Qwen/Qwen3-0.6B',
 'response_source': 'model',
 'task': 'Найди номер аудитории следующего занятия.',
 'context': 'Доступен только lookup(key). Ключи: room — аудитория; time — начало занятия; teacher — преподаватель. Значения можно узнать только через инструмент. Формат ответа: {"tool":"lookup","args":{"key":"KEY"}}, где KEY — выбранный ключ. Верни только JSON, без Markdown.',
 'raw_response': '{"tool":"lookup","args":{"key":"room"}}',
 'response_received': True,
 'json_valid': True,
 'action_valid': True,
 'selected_key': 'room',
 'key_correct': True,
 'found': True,
 'result_correct': True,
 'error': None}

In [73]:
record_attempt("Кто ведет это занятие?", tool_context)

{'attempt': 2,
 'backend': 'local',
 'model_id': 'Qwen/Qwen3-0.6B',
 'response_source': 'model',
 'task': 'Кто ведет это занятие?',
 'context': 'Доступен только lookup(key). Ключи: room — аудитория; time — начало занятия; teacher — преподаватель. Значения можно узнать только через инструмент. Формат ответа: {"tool":"lookup","args":{"key":"KEY"}}, где KEY — выбранный ключ. Верни только JSON, без Markdown.',
 'raw_response': '{"tool":"lookup","args":{"key":"teacher"}}',
 'response_received': True,
 'json_valid': True,
 'action_valid': True,
 'selected_key': 'teacher',
 'key_correct': False,
 'found': True,
 'result_correct': False,
 'error': None}

In [74]:
record_attempt("В каком здании находится аудитория?", tool_context)

{'attempt': 3,
 'backend': 'local',
 'model_id': 'Qwen/Qwen3-0.6B',
 'response_source': 'model',
 'task': 'В каком здании находится аудитория?',
 'context': 'Доступен только lookup(key). Ключи: room — аудитория; time — начало занятия; teacher — преподаватель. Значения можно узнать только через инструмент. Формат ответа: {"tool":"lookup","args":{"key":"KEY"}}, где KEY — выбранный ключ. Верни только JSON, без Markdown.',
 'raw_response': '{"tool":"lookup","args":{"key":"room"}}',
 'response_received': True,
 'json_valid': True,
 'action_valid': True,
 'selected_key': 'room',
 'key_correct': True,
 'found': True,
 'result_correct': True,
 'error': None}

In [71]:
# Необязательно: раскомментируйте одну или две строки после изменения формулировки.
# В mock меняйте scripted_response, чтобы исследовать заранее выбранные ошибки.
# record_attempt("Узнай аудиторию, используя доступный инструмент.", tool_context,
#                scripted_response='{"tool":"lookup","args":{"key":"building"}}')
# record_attempt("Какой ключ нужно запросить для номера аудитории?", tool_context,
#                scripted_response='не JSON')

In [75]:
print("Журнал учебной задачи:")
print(json.dumps(experiment_records, ensure_ascii=False, indent=2))
print("Технический журнал клиента (включает диагностику):")
print([] if client is None else client.records)

Журнал учебной задачи:
[
  {
    "attempt": 1,
    "backend": "local",
    "model_id": "Qwen/Qwen3-0.6B",
    "response_source": "model",
    "task": "Найди номер аудитории следующего занятия.",
    "context": "Доступен только lookup(key). Ключи: room — аудитория; time — начало занятия; teacher — преподаватель. Значения можно узнать только через инструмент. Формат ответа: {\"tool\":\"lookup\",\"args\":{\"key\":\"KEY\"}}, где KEY — выбранный ключ. Верни только JSON, без Markdown.",
    "raw_response": "{\"tool\":\"lookup\",\"args\":{\"key\":\"room\"}}",
    "response_received": true,
    "json_valid": true,
    "action_valid": true,
    "selected_key": "room",
    "key_correct": true,
    "found": true,
    "result_correct": true,
    "error": null
  },
  {
    "attempt": 2,
    "backend": "local",
    "model_id": "Qwen/Qwen3-0.6B",
    "response_source": "model",
    "task": "Кто ведет это занятие?",
    "context": "Доступен только lookup(key). Ключи: room — аудитория; time — начало за

## 8. Отчёт · 2 балла
Сохраните выполненный notebook с реализациями и выводами открытых проверок. Заполните следующую текстовую ячейку. Оценивается корректная интерпретация, включая честное описание неудачного или невыполненного live-запуска.

1. Какой backend и model_id использовались? Выполнялась ли реальная генерация?
2. Сколько попыток сделано? Для каждой разделите получение ответа, корректность JSON, допустимость действия и правильность результата. Используйте журнал, не придумывайте результаты.
3. Где решение выбирала модель, а где работал обычный код? Какие части задавал человек?
4. Почему одна успешная попытка не доказывает надёжность агента? Какой случай вы добавили бы в проверку?

**Для желающих, без дополнительных баллов.** Верните наблюдение модели и попросите сформировать финальный ответ; запланируйте этот дополнительный вызов в пределах трёх модельных попыток и бюджета клиента. Объясните, чем этот шаг отличается от чтения словаря. Это расширение не требуется для сдачи.

**Мой отчёт:**

1. Использовалась локальная модель `Qwen/Qwen3-0.6B`, ответы создавались по-настоящему.

2. Из 3 попыток формат JSON всегда был верным. Но код проверки оказался с ошибкой: он жестко ожидал только ответ `room`. Поэтому во второй попытке он забраковал правильный ответ модели, а в третьей — ошибочно засчитал промах нейросети как успех.

3. Нейросеть сама выбирала, что искать. Код проверял текст и искал данные. Человек задал правила.

4. Одна попытка не доказывает надежность агента. Третий тест показал, что нейросеть может просто угадывать ответ. Нужно исправить проверочный код и добавить вопросы с подвохом.